STOPWORDS E RIMOZIONE RUMORE

Stopwords e rimozione rumore fanno parte del preprocessing del testo, cioè delle operazioni che fai prima di passare i dati al modello.
testo grezzo -> pulizia/rimozione rumore -> tokenizzazione -> envetuale rimozione stopwords -> rappresentazione numerica -> modello.

Immagina do voler ascoltare una melodia in una stanza con gente che sussura continuamente, per ascoltare dovresti chiedere di fare silenzione. Questo è quello che si intendo con rimozione stopwords e rumore, eliminazione del brusio di fondo

Ma cosa intendiamo esatttamente per brusio di fondo in un testo? 

Tutte le parole, hanno lo stesso peso?
Stopwords: Filtrare l'Essenziale
Comprendere il rumore statistico nel linguaggio naturale
In ogni lingua esistono parole che fungono da collante grammaticale ma che non portano un significato semantico distintivo. Queste parole, come articoli, preposizioni e congiunzioni, compaiono con una frequenza elevatissima in quasi ogni documento.
Nel Deep Learning per NLP, mantenere queste parole può aumentare inutilmente la dimensione delle spazio vettoriale, quandi aumenta il costo, senza migliorare la capacità del modello di distinguere tra diversi argomenti o sentimenti.

Le StopWords sono parole molto frequenti che spesso portano poca informazione, anche se grammaticalmente corrette.
In italiano sono: il, lo, la, gli, le, di, a, da,  in, con, su, per, e, o, che, un, una

Nella frase "il cliente ha ordinato 100 pezzi" dopo la rimozione delle stopwords la frase diventa "cliente ordinato 100 pezzi"

La frequenza di una parole è inversamente proporzionale al suo rango nella classifica di frequenza, questa è la legge di Zipf. Meno parole è probabile più informazione ci fornisce.
Le stopword occupano spesso il 40-50% di un intero documento. Meno token significano meno memoria RAM utilizzazata e tempi di addestramento più veloci

La rimozione delle stopwords era molto comune nel NLP classico, con Trasformer e LLM normalmente non rimuovi le stopword perchè?
perchè pensa alla frase "la merca non è disponibile" se rimuovessi "non" avresti distrutto vero significato della frase.
Eventualmente potesti eliminare parole che sono considerate stopwords dall'elenco delle stopwords

Per eliminazione rumore si intende eliminazione di parte di testo che non sono utili al compito.
Per esempio in una mail, se l'obbiettivo è estrarre i dati di un eventuale ordine (codice articolo, quantità e data consegna, emoji) la firma, il telefono, url, e tutta la catena di mail precedente, sono solo rumore, non aggiungono significato. Quindi potresti "pulire" la mail di questo rumore prima di classificarla.

I dati di testo reale sono spesso sporchi. Se scarichiamo dati da siti web o social media, trovereom tag HTML, URL, simboli speciali e sequenze di caratteri che non appartengono alla lingua naturale.
Il rumore non è solo estestico, per un computer "python" o "python!" sono token distinti che frammentano l'apprendimento. Se non puliamo il nostro modello questo pensarà di avere due concetti diversi.

Attenzione perchè anche la pulizia dal rumore deve essere domain-aware, cioè consapevole al dominio
Esempio nella mail: "trasmetto ordine abc-123" "abc-123" dopo eliminazione rumore potrebbe diventare "abc123" ma questo cambia l'identificativo dell'ordine.

Un'altro aspetto è il lowercasing, cioè trasformare tutto in minuscolo (testo.lower())
Ma anche qui: dipende dal contesto. In alcuni casi (vedi anche Python) ABC123 è differente da abc123. oppure modelli cased come "bert-base-italian-xxl-cased" sono stati addestrati mantenendo maiuscole/minuscole, quindi non dovresti necessariamente trasformare tutto in minuscolo prima di passare il testo al modello.

Elementi di Disturbo Comuni
Cosa dobbiamo eliminare prima delle tokenizzazione?
* Tag HTML e XML: residui di scraping come 'div', 'br' o entità come '&amp;'
* Url e Ling: indirizzi web che non portano valore semantico generale ma occupano spazio nel vocabolario
* Caratteri speciali: simboli matematici, emoji (se non rilevanti) e punteggiatura eccessiva
* Normalizzazione: conversione di tutto il testo in minuscolo (folding) per unificare i token.

Ma come facciamo, operativamente, a eliminare questo rumore?

Strumenti di Pulizia
- Espressioni Regolare (Regex): il tool principale per la pulizia è il modulo 're' di Python, che permette di definire pattern complessi per indivisuare e sostituire stringa.
- Parsing HTML: librerie come BeautifulSoup sono preferibili alle Regex per rimuovere tag nidificati e più complessi, in modo sicuro e robusot
- Gestione Unicode: rimuovere o convertire caratteri accentati o simboli non-ASCII per mantenere la consistenza del set di caratteri

Approfondimento: Espressioni Regolari
Logica dei Pattern
Una Regex è un linguaggio formale per descrivere insieme di stringhe. Ad esempio, per catturare un URL, definiamo una sequenza che inizia ocn "http" seguita da caratteri non spaziati
Sia "S" l'insieme di tutte le stringhe, la pulizia tramite Regex definisce uan funzione matematica di trasformazione "f" che mappa stringhe sporche in stringhe pulite.
Se definisco bene la mia regex, posso trasformare migliaia di pagine web caotiche, in un testo cristallino in pochi secondi.

Implementazione in Python
In Python usare sempre i set ma le liste (list), perchè cercare una parola in un set è costante, mentre in una lista il tempo aumenta in base alla dimensione della lista stessa.
In spaCy è possibile aggiungere "is_stop" in ogni token, rendendo il filtraggio parte integrante del processi di parsing.
E' buona norma visualizzare le parole più frequenti dopo la pulizia per verificare se sono rimasti termini indesiderati. 

Approfondimento: L'impatto delle Negazioni
Il problema del "not"
Rimuovere ciecamente lo stopword può essere catastrofico. Considerate la frase: "non è un buon film", rimuovendo "non" e "è", resta "buon film", trasformando una critica negativa in positiva
Dobbiamo sempre bilanciare la puliza con la preservazione del significato specialmente quando usiamo modelli che non considerano l'ordine delle parole (Bag of Words)

Nel NLP classico pulisci molto, ma con i Transformer/LLM questo non è più vero, pulisci solo ciò che sei sicuro che sia rumore.


In [1]:
import os

# Configurazione del Backend per Keras 3 (Best Practice 2026)
# Impostiamo PyTorch come motore computazionale prima di caricare Keras
os.environ["KERAS_BACKEND"] = "torch"

import keras
import re
import nltk
from nltk.corpus import stopwords
from typing import List, Set

# Download delle risorse necessarie (eseguito solo la prima volta)
# NLTK rimane lo standard accademico per le liste di stopwords multilingua
nltk.download('stopwords')

def clean_text_pipeline(raw_text: str, custom_stops: List[str] = []) -> str:
    """
    Pipeline completa di pulizia e rimozione del rumore.
    
    Teoria: Il 'Rumore' è ogni informazione che non contribuisce alla 
    distribuzione semantica del testo. Rimuoverlo riduce la varianza dei dati.
    """
    
    # 1. Normalizzazione (Case Folding)
    # Fondamentale per evitare che 'Python' e 'python' siano visti come token diversi
    text = raw_text.lower()
    
    # 2. Rimozione Tag HTML
    # Teoria: I tag (es. <div>) sono rumore strutturale, non linguistico.
    # Usiamo una Regex che identifica tutto ciò che è racchiuso tra < e >
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Rimozione URL
    # Gli URL hanno entropia altissima ma valore semantico nullo in compiti generici.
    # Pattern per catturare http, https e www
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. Rimozione caratteri speciali e punteggiatura
    # Manteniamo solo lettere e spazi. Nota: in certi casi (es. Sentiment Analysis)
    # i punti esclamativi potrebbero essere mantenuti. Qui puliamo tutto.
    text = re.sub(r'[^a-zàèìòù\s]', '', text)
    
    # 5. Gestione Stopwords
    # Carichiamo le stopwords italiane standard
    stop_words_set: Set[str] = set(stopwords.words('italian'))
    
    # Personalizzazione: Aggiunta di termini specifici del dominio (Customizzazione)
    # Teoria: Le liste standard sono generaliste; ogni dataset ha il suo rumore specifico.
    if custom_stops:
        stop_words_set.update(custom_stops)
    
    # Eccezione: Rimuoviamo 'non' dalle stopwords se vogliamo preservare la negazione
    # Teoria: In NLP 'non' è spesso una stopword, ma è vitale per il senso logico.
    if 'non' in stop_words_set:
        stop_words_set.remove('non')
    
    # 6. Tokenizzazione e Filtraggio
    # Dividiamo per spazi e rimuoviamo i termini se presenti nel set (Lookup O(1))
    tokens = text.split()
    cleaned_tokens = [w for w in tokens if w not in stop_words_set]
    
    # Ricostruiamo la stringa pulita
    return " ".join(cleaned_tokens)

# --- ESEMPIO DI UTILIZZO ---

raw_data = [
    "Il corso di AI è fantastico! <br> Visita https://ai-deeplearning.it per info.",
    "Non mi è piaciuto il modulo, troppo complesso e pieno di bug.",
    "L'intelligenza artificiale (AI) cambierà il mondo! 🤖 #AI2026"
]

# Definiamo stopwords specifiche per il nostro dominio (es. 'corso', 'modulo')
my_custom_stops = ['corso', 'modulo', 'info', 'ai']

print("--- Inizio Processamento Testi ---")
for doc in raw_data:
    clean_doc = clean_text_pipeline(doc, custom_stops=my_custom_stops)
    print(f"Originale: {doc}")
    print(f"Pulito:    {clean_doc}\n")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\barbara\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


--- Inizio Processamento Testi ---
Originale: Il corso di AI è fantastico! <br> Visita https://ai-deeplearning.it per info.
Pulito:    fantastico visita

Originale: Non mi è piaciuto il modulo, troppo complesso e pieno di bug.
Pulito:    non piaciuto troppo complesso pieno bug

Originale: L'intelligenza artificiale (AI) cambierà il mondo! 🤖 #AI2026
Pulito:    lintelligenza artificiale cambierà mondo

